# 1. Introduction

This project builds a **BST-Backed Hyperparameter Optimiser with Transfer Analysis** — a complete data science tool that uses a hand-rolled **Binary Search Tree (BST)** as the storage layer for hyperparameter trial results.

**Phase 1** runs an exhaustive grid search on the *Breast Cancer Wisconsin* dataset, storing every result in a BST keyed by cross-validated accuracy.  
**Phase 2** reloads the BST, re-evaluates every configuration on the *Banknote Authentication* dataset, rebuilds the tree, and analyses which hyperparameter configurations transfer well between domains.

The project deliberately separates *algorithm* from *ML* — the BST does not know about machine learning, and the ML layer does not know about tree internals. This separation makes every module independently testable and reusable.

### Course concepts map

| Concept | Session | Module | Complexity |
|---|---|---|---|
| Brute-force exhaustive search (`itertools.product`) | 1 | `grid_search.py` | O(n) combinations |
| Hash tables for O(1) rank lookup | 1 | `transfer.py` | O(1) per lookup |
| `functools.wraps` — decorator pattern | 1 | `timer.py` | — |
| Recursion & divide-and-conquer | 2 | `bst.py`, `rebuild.py` | O(log n) stack |
| Comparison-based sorting | 3 | `transfer.py` | O(n log n) |
| Why insertion order matters | 3 | `rebuild.py` | O(n²) worst case |
| Binary search & range queries | 4 | `registry.py` | O(k + h) |
| BST traversals | 5 | `bst.py` | O(n) each |
| BST balance & height | 5 | `registry.py`, `rebuild.py` | O(n) check |

> **AI Tools disclosure:** Claude (Anthropic) was used as a coding accelerator for boilerplate and docstrings. All algorithmic logic in `bst_toolkit` was written and is understood by the author.


## 1.1 Package Imports

In [7]:
# Setting up Python path for imports
import sys
sys.path.insert(0, '/Users/arman/Documents/Project/BST-Backed-Hyperparameter-Optimiser-with-Transfer-Analysis')

# Importing python libraries

import os, math, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

#----------------------------------------------------------------------------

# Importing project modules

from capstone_project.bst_toolkit.registry import HyperparamRegistry
from capstone_project.bst_toolkit.rebuild  import (
    rebuild_naive, rebuild_shuffled, rebuild_balanced
)
from capstone_project.ml_toolkit.grid_search import grid_search
from capstone_project.ml_toolkit.transfer    import analyse_transfer, transfer_summary
from capstone_project.benchmarks.timer       import timed, benchmark, benchmark_report

from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA

# Setting up the environment

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)
print("All packages imported ✓")

All packages imported ✓


# 2. Dataset A — Breast Cancer Wisconsin (Diagnostic)

The **Breast Cancer Wisconsin (Diagnostic)** dataset contains **569 tissue samples**, each described by **30 real-valued measurements** of cell nuclei geometry extracted from digitised fine-needle aspirate (FNA) images.

The **target** is binary: **malignant (M → 1)** vs **benign (B → 0)**.

Each of the 10 geometric properties — *radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension* — is reported as three statistics:

| Statistic | Meaning |
|---|---|
| `_mean` | Mean value across all nuclei in the image |
| `_se` | Standard error — variability of measurements |
| `_worst` | Largest value (most extreme nucleus) |

This gives 3 × 10 = **30 features** total.


## 2.1 Download / Verify

1.   List item
2.   List item



In [9]:
import subprocess
result = subprocess.run(
    [sys.executable, "capstone_project/data/download.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)



STDERR: /Users/arman/Documents/Project/BST-Backed-Hyperparameter-Optimiser-with-Transfer-Analysis/.venv/bin/python: can't open file '/Users/arman/Documents/Project/BST-Backed-Hyperparameter-Optimiser-with-Transfer-Analysis/capstone_project/notebook/capstone_project/data/download.py': [Errno 2] No such file or directory



## 2.2 Load & Quality Check

In [11]:
WDBC_PATH = "capstone_project/ript/wdbc.csv"
df_a = pd.read_csv(WDBC_PATH)
print(f"Shape: {df_a.shape}")

# Check for missing values
missing = df_a.isnull().sum()
print(f"Missing values: {missing.sum()} (across all columns)")
df_a.head()


FileNotFoundError: [Errno 2] No such file or directory: 'capstone_project/ript/wdbc.csv'

In [ ]:
# Descriptive statistics — split by class for comparison
feat_cols_a = [c for c in df_a.columns if c != 'diagnosis']
desc = df_a.groupby('diagnosis')[feat_cols_a[:10]].describe().T
desc.columns = ['_'.join(map(str,c)) for c in desc.columns]
desc.round(3)


> **Observation:** Even without visualisation, the `mean` rows show that malignant samples (1) have consistently larger values for radius, perimeter, area, concavity and concave_points than benign samples (0). These differences are the statistical signal our classifier will learn to exploit.